## 0. Environment Setup

In [1]:
import os
from dotenv import load_dotenv
load_dotenv(dotenv_path="../.env", override=True)

# ROS / ament package paths needed for PR2 xacro parsing.
# These must be set BEFORE any pycram/SDT imports.
_IAI_PR2_INSTALL = "/home/malineni/ros_packages/iai_pr2_install"
_IAI_PR2_SRC     = "/home/malineni/ros_packages/iai_pr2/iai_pr2_description"
os.environ["AMENT_PREFIX_PATH"] = _IAI_PR2_INSTALL + ":" + os.environ.get("AMENT_PREFIX_PATH", "")
os.environ["ROS_PACKAGE_PATH"]   = _IAI_PR2_SRC     + ":" + os.environ.get("ROS_PACKAGE_PATH", "")

print("Environment configured.")

Environment configured.


In [2]:
from langchain_openai import ChatOpenAI
from agentic_llmr.core.orchestrator import ReActAgent
from agentic_llmr.integrations.world_manager import set_active_world
from uniworld import load_pr2_apartment_world
from agentic_llmr.core.trace import render_trace

print("Imports successful.")

`polytope` failed to import `cvxopt.glpk`.
will use `scipy.optimize.linprog`


Imports successful.


## 1. Load the World

We use `load_pr2_simple_world()` from `uniworld`, which builds a self-contained SDT world:
- **PR2** robot (fully articulated, 40+ joints)
- **Table** (URDF) at x=1.1 m
- **Milk carton** (STL mesh) at (0.9, 0.0, 0.85) with `Milk` semantic annotation
- **Cereal box** (STL mesh) at (0.9, 0.2, 0.85) with `Cereal` semantic annotation

In [3]:
print("Loading PR2 simple world...")
world, robot_view, context = load_pr2_apartment_world()
set_active_world(world, robot_view)

bodies      = list(world.bodies)
annotations = list(world.semantic_annotations)

print(f"\nWorld loaded successfully!")
print(f"  Bodies              : {len(bodies)}")
print(f"  Semantic annotations: {len(annotations)}")
print("\nSemantic annotations:")
for ann in annotations:
    bodies_names = [str(getattr(b.name, "name", b.name)) for b in getattr(ann, "bodies", [])]
    print(f"  [{ann.__class__.__name__}]  bodies={bodies_names}")


Loading PR2 simple world...


Unknown attribute "type" in /robot[@name='pr2']/link[@name='base_laser_link']
Unknown attribute "type" in /robot[@name='pr2']/link[@name='wide_stereo_optical_frame']
Unknown attribute "type" in /robot[@name='pr2']/link[@name='narrow_stereo_optical_frame']
Unknown attribute "type" in /robot[@name='pr2']/link[@name='laser_tilt_link']
Unknown tag "material" in /robot[@name='pr2']/link[@name='l_force_torque_link']/collision[1]
Unknown tag "material" in /robot[@name='apartment']/link[@name='coffe_machine']/collision[1]



World loaded successfully!
  Bodies              : 221
  Semantic annotations: 15

Semantic annotations:
  [Finger]  bodies=['l_gripper_l_finger_link', 'l_gripper_l_finger_tip_link']
  [Finger]  bodies=['l_gripper_r_finger_link', 'l_gripper_r_finger_tip_link']
  [ParallelGripper]  bodies=['l_gripper_motor_accelerometer_link', 'l_gripper_r_finger_tip_link', 'l_gripper_l_finger_tip_link', 'l_gripper_r_finger_link', 'l_gripper_led_frame', 'l_gripper_l_finger_tip_frame', 'l_gripper_l_finger_link', 'l_gripper_palm_link', 'l_gripper_motor_screw_link', 'l_gripper_motor_slider_link', 'l_gripper_tool_frame']
  [Arm]  bodies=['torso_lift_link', 'l_shoulder_pan_link', 'l_shoulder_lift_link', 'l_upper_arm_roll_link', 'l_upper_arm_link', 'l_elbow_flex_link', 'l_forearm_roll_link', 'l_forearm_link', 'l_wrist_flex_link', 'l_wrist_roll_link']
  [Finger]  bodies=['r_gripper_l_finger_link', 'r_gripper_l_finger_tip_link']
  [Finger]  bodies=['r_gripper_r_finger_link', 'r_gripper_r_finger_tip_link']
  [P

## 2. Initialise the Agent

The `ReActAgent` (Orchestrator) delegates to three specialist sub-agents:
- **SceneQueryAgent** — 16 SDT perception tools (poses, surfaces, spatial relations, placement)
- **KinematicsAgent** — Giskardpy reachability and grasp-pose tools
- **PlanningAgent** — PyCRAM action schema discovery and physics simulation

In [4]:
llm   = ChatOpenAI(model="gpt-4o-mini", temperature=0)
agent = ReActAgent(llm=llm)

print(f"Orchestrator ready — {len(agent.tools)} top-level tools (delegates to 3 sub-agents):\n")
for t in agent.tools:
    print(f"  • {t.name}")

print(f"\nSub-agent tool breakdown:")
print(f"  SceneQueryAgent : {len(agent._scene_query_agent.tools)} tools")
print(f"  KinematicsAgent : {len(agent._kinematics_agent.tools)} tools")
print(f"  PlanningAgent   : {len(agent._planning_agent.tools)} tools  (schema discovery + simulation)")


Orchestrator ready — 5 top-level tools (delegates to 3 sub-agents):

  • write_scratchpad
  • read_scratchpad
  • query_scene_perception
  • query_kinematics
  • query_action_schema

Sub-agent tool breakdown:
  SceneQueryAgent : 30 tools
  KinematicsAgent : 11 tools
  PlanningAgent   : 5 tools  (schema discovery + simulation)


## 3. Query Helper

A small utility that:
1. **Clears the scratchpad** so each query starts fresh
2. **Runs the agent** and prints the live reasoning trace
3. Returns the agent's final response

In [5]:
import os

def run_query(instruction: str, hint: str = "") -> str:
    """Submit an instruction to the agent and display the full reasoning trace."""
    for pad in [
        "orchestrator_scratchpad.md", "sdt_scratchpad.md",
        "giskard_scratchpad.md", "pycram_scratchpad.md",
    ]:
        with open(pad, "w") as f:
            f.write("")

    print(f"\n{chr(9473)*65}")
    print(f"  QUERY: {instruction}")
    print(f"{chr(9473)*65}\n")

    response = agent.resolve_action(instruction=instruction, template_context=hint)

    print(f"\n{chr(9472)*65}")
    print("FINAL RESPONSE:")
    print(response[:3000] if len(response) > 3000 else response)
    print(f"{chr(9472)*65}\n")
    return response

print("Query helper ready.")


Query helper ready.


---
## Query 1 — Scene Overview

**Goal:** Get a complete picture of everything in the world.

**Expected SDT tools:** `get_objects_in_scene`, `list_semantic_annotations`

In [6]:
r1 = run_query(
    "Give me a complete overview of the current scene. "
    "List all objects present, their semantic types, and their 3D positions."
)


━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
  QUERY: Give me a complete overview of the current scene. List all objects present, their semantic types, and their 3D positions.
━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━


--- [ORCHESTRATOR STARTED] ---
[Orchestrator]:
Instruction: Give me a complete overview of the current scene. List all objects present, their semantic types, and their 3D positions.
Context: 

Please resolve this into executable parameters.

[Delegate] → query_scene_perception(['query'])

  ► [Scene Agent] List all objects present, their semantic types, and their 3D positions.
[SDT Tool] Querying native semantic scene graph...
  ◄ [Scene Agent] Done.

[Orchestrator]:
The following objects are present in the scene:

1. **Milk**
   - **Position:** (2.370, 2.000, 1.050)

2. **Cereal**
   - **Position:** (2.370, 1.800, 1.050)

3. **Robot**
   - **Position:** (1.300, 2.000, 0.051)

[Orchestrator]:
The current scene contains the f

In [7]:
render_trace(agent.last_trace, title="Query 1 — Scene Overview",
               output_path="outputs/q1_scene_overview.html", open_browser=False)

Trace saved → /home/malineni/workingdir/cognitive_robot_abstract_machine/agentic_llmr/notebooks/outputs/q1_scene_overview.html


---
## Query 2 — Finding Objects by Type

**Goal:** Locate specific semantic categories in the scene.

**Expected SDT tools:** `find_objects_by_type`, `get_object_pose`

In [ ]:
r2 = run_query(
    "Find all food-related objects in the scene. "
    "Report the exact 3D position of each one."
)

---
## Query 3 — Surface Query

**Goal:** Understand which objects are on a specific surface.

**Expected SDT tools:** `list_semantic_annotations`, `get_objects_on_surface`

In [8]:
r3 = run_query(
    "Which objects are currently resting on the table? "
    "Report each object's semantic type, body name, and position."
)


━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
  QUERY: Which objects are currently resting on the table? Report each object's semantic type, body name, and position.
━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━


--- [ORCHESTRATOR STARTED] ---
[Orchestrator]:
Instruction: Which objects are currently resting on the table? Report each object's semantic type, body name, and position.
Context: 

Please resolve this into executable parameters.

[Delegate] → query_scene_perception(['query'])

  ► [Scene Agent] Which objects are currently resting on the table? Report each object's semantic type, body name, and position.
[SDT Tool] Querying objects on surface: Table
[SDT Tool] Listing semantic annotations...
[SDT Tool] Searching for semantic type: Cereal
[SDT Tool] Searching for semantic type: Milk
  ◄ [Scene Agent] Done.

[Orchestrator]:
The following objects are currently resting on the table:

1. **Cereal**
   - **Body Name:** breakfast_cereal.stl
 

In [9]:
render_trace(agent.last_trace, title="Query 3 — Surface query",
               output_path="outputs/q3_surface_query.html", open_browser=False)

Trace saved → /home/malineni/workingdir/cognitive_robot_abstract_machine/agentic_llmr/notebooks/outputs/q3_surface_query.html


---
## Query 4 — Object Property Analysis

**Goal:** Inspect the physical characteristics of a specific object.

**Expected SDT tools:** `get_object_pose`, `get_object_dimensions`, `get_object_orientation`

In [ ]:
r4 = run_query(
    "Analyse the milk carton in detail: "
    "What is its exact 3D pose? What are its physical dimensions (width, depth, height)? "
    "Is it upright or lying on its side?"
)

---
## Query 5 — Spatial Relationship Reasoning

**Goal:** Understand the relative positions between objects and the robot.

**Expected SDT tools:** `get_spatial_relation`, `get_nearest_objects`

In [ ]:
r5 = run_query(
    "What is the spatial relationship between the milk carton and the cereal box? "
    "Which of the two is closer to the robot, and by how much?"
)

---
## Query 6 — Size Comparison and Disambiguation

**Goal:** Rank objects by physical size — needed to resolve instructions like *'the larger object'*.

**Expected SDT tools:** `sort_objects_by_size`, `get_object_dimensions`

In [ ]:
r6 = run_query(
    "Compare the physical sizes of the milk carton and the cereal box. "
    "Which one is larger? Provide their dimensions to justify your answer."
)

---
## Query 7 — Placement Planning

**Goal:** Find a suitable surface and free spots to place an object.

**Expected SDT tools:** `get_objects_on_surface`, `get_best_surface_for_placement`, `get_free_placement_spots`

In [10]:
r7 = run_query(
    "I want to place the cereal box somewhere after picking it up. "
    "Find the best surface available and suggest at least 3 concrete (x, y, z) positions "
    "on that surface where I could place it."
)


━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
  QUERY: I want to place the cereal box somewhere after picking it up. Find the best surface available and suggest at least 3 concrete (x, y, z) positions on that surface where I could place it.
━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━


--- [ORCHESTRATOR STARTED] ---
[Orchestrator]:
Instruction: I want to place the cereal box somewhere after picking it up. Find the best surface available and suggest at least 3 concrete (x, y, z) positions on that surface where I could place it.
Context: 

Please resolve this into executable parameters.

[Delegate] → query_scene_perception(['query'])

  ► [Scene Agent] Find the best surface available for placing an object and provide its details.
[SDT Tool] Querying native semantic scene graph...
[SDT Tool] Classifying scene objects by role...
[SDT Tool] Listing semantic annotations...
[SDT Tool] Querying objects on surface: Table
[SDT Tool] Querying objects on 

In [11]:
render_trace(agent.last_trace, title="Query 7 — Surface query",
               output_path="outputs/q7_placement.html", open_browser=False)

Trace saved → /home/malineni/workingdir/cognitive_robot_abstract_machine/agentic_llmr/notebooks/outputs/q7_placement.html


---
## Query 8 — Full Pick-Up Parameter Resolution

**Goal:** Exercise the complete multi-tool reasoning pipeline for a manipulation task.

**Expected tools (in order):**
1. `write_scratchpad` — document plan
2. `list_available_actions` + `get_action_documentation` — discover `PickUpAction` schema
3. `find_objects_by_type` / `get_object_pose` — locate the milk
4. `get_object_dimensions` + `get_object_orientation` — understand object geometry
5. `check_kinematic_reachability` — verify arm can reach it
6. `get_grasp_poses` — get valid grasp descriptions

> **Note:** Execution (`simulate_action`) is intentionally skipped here. The agent will output the resolved JSON parameters.


In [ ]:
r8 = run_query(
    "Pick up the milk carton from the table.",
    hint="Resolve all parameters for a PickUpAction. Do not simulate — just output the final JSON."
)

---
## Query 9 — Multi-Constraint Reasoning

**Goal:** The agent must combine spatial, reachability, and size reasoning to make a decision.

**Expected tools:** `get_objects_on_surface`, `get_spatial_relation`, `get_object_dimensions`, `check_kinematic_reachability`

In [ ]:
r9 = run_query(
    "There are two objects on the table. "
    "Considering their positions relative to the robot and which arm can reach them, "
    "which object should I pick up first and with which arm? Justify your answer."
)

---
## Query 10 — Robot Configuration Inspection

**Goal:** Read the current robot joint states — useful before planning arm movements.

**Expected tools:** `get_joint_states`, `list_semantic_annotations`

In [ ]:
r10 = run_query(
    "What is the current configuration of the PR2 robot? "
    "Report the current joint positions for both arms and summarise whether the arms "
    "are in a parked, extended, or arbitrary configuration."
)

In [ ]:
r11 = run_query("does the robot hold anything?")

---
## Summary: SDT Tool Coverage

| Query | Primary tools exercised |
|---|---|
| 1 — Scene overview | `get_objects_in_scene`, `list_semantic_annotations` |
| 2 — Find by type | `find_objects_by_type`, `get_object_pose` |
| 3 — Surface query | `get_objects_on_surface` |
| 4 — Object properties | `get_object_pose`, `get_object_dimensions`, `get_object_orientation` |
| 5 — Spatial relations | `get_spatial_relation`, `get_nearest_objects` |
| 6 — Size comparison | `sort_objects_by_size`, `get_object_dimensions` |
| 7 — Placement planning | `get_best_surface_for_placement`, `get_free_placement_spots` |
| 8 — Full pick-up plan | All SDT + PyCRAM + Giskard tools in sequence |
| 9 — Multi-constraint | `get_spatial_relation`, `check_kinematic_reachability`, `get_object_dimensions` |
| 10 — Robot config | `get_joint_states` |